# L12 — Python Data Tools (Tutorial)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L12/00_python_data_tools/python_data_tools_starter.ipynb)

**Read this first if you're new to pandas / numpy / scipy.** It's a 30-minute tour of the specific functions used by the L11 + L12 starter notebooks. Every cell stands on its own — no external CSVs, no hardware, just tiny made-up data so you can change a value and see what happens.

**How to use:** press `Shift + Enter` on each cell, top to bottom. Read the markdown, run the code, *then* go open one of the project starters.

---
# Section 1 — pandas: tables you can manipulate

A **DataFrame** is a table with named columns — like a spreadsheet you can manipulate from Python. Every starter notebook loads a CSV into a DataFrame as the first real step.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name":  ["alice", "bob", "carla", "dan"],
    "score": [82, 91, 70, 88],
})
df

## `pd.read_csv` — read a CSV into a DataFrame

In real notebooks the path is a file or URL. Here we fake one with `StringIO` so the cell is self-contained.

In [ ]:
from io import StringIO

csv_text = """t,x,y,z
0.0,10,20,1000
0.1,15,18,995
0.2,8,22,1002
0.3,12,19,998
0.4,11,21,1001
"""
df = pd.read_csv(StringIO(csv_text))
df.head(3)        # first 3 rows; .head() defaults to 5

## Picking columns and rows

- **Column** by name: `df["x"]` returns one column (a `Series`). `df[["x", "y"]]` returns multiple columns (a `DataFrame`).
- **Rows** by position: `df.iloc[start:stop]` slices rows — this is how the windowing loops in L11 / Project 6 walk through a recording.

In [ ]:
print("x column:")
print(df["x"])

print("\nrows 1..3 (positions 1, 2):")
df.iloc[1:3]

## Per-column statistics

Every column knows its own `mean()`, `std()`, `max()`, `min()`. The L11/L12 feature extractors use exactly these.

In [ ]:
print("mean of x:", df["x"].mean())
print("std  of x:", df["x"].std())
print("max  of x:", df["x"].max())
print("min  of x:", df["x"].min())

## `.apply(fn)` — run a function over every value

Used in L11 / Project 2 to label rows: hand `apply` a function, get a new column.

In [ ]:
def bucket(t):
    if t < 0.2: return "early"
    if t < 0.4: return "middle"
    return "late"

df["phase"] = df["t"].apply(bucket)
df

## `.value_counts()` — how many of each category?

After labeling, students often want to confirm their classes are balanced. `value_counts()` is the one-liner.

In [ ]:
df["phase"].value_counts()

## Building a DataFrame from a list of dicts

The L11 windowing loop appends one dict per window, then turns the list of dicts into a DataFrame. Same shape as our starter `df`.

In [ ]:
rows = []
for i in range(3):
    rows.append({"window": i, "mean_mag": 100 + i*10, "std_mag": 5 + i})
features = pd.DataFrame(rows)
features

---
# Section 2 — numpy: fast math on arrays

A numpy **array** is like a Python list, but operations work on the *whole array at once*. Element-wise math is the bread and butter.

In [ ]:
import numpy as np

x = np.array([3, 4, 0, -3])
y = np.array([4, 3, 5,  4])

print("x + y      =", x + y)
print("x * 10     =", x * 10)
print("x**2 + y**2=", x**2 + y**2)        # element-wise square then add
print("sqrt of that:", np.sqrt(x**2 + y**2))   # this is exactly what "magnitude" computes

## `np.diff` — gaps between consecutive values

Used in the **beat tracker** to convert a list of beat *times* into the *intervals* between them — `60 / mean(intervals)` = BPM.

In [ ]:
beat_times = np.array([0.50, 1.00, 1.55, 2.05, 2.60])
intervals  = np.diff(beat_times)            # length is one shorter
print("intervals:", intervals)              # [0.5, 0.55, 0.5, 0.55]
print("mean interval:", intervals.mean())
print("BPM:          ", 60 / intervals.mean())

## `np.arange` — generate a regular sequence

Like Python's `range()`, but returns a numpy array (so you can do math on it). Used in the synchrony notebook to build a list of *lags* for cross-correlation.

In [ ]:
print(np.arange(0, 1, 0.2))         # 0.0, 0.2, 0.4, 0.6, 0.8
print(np.arange(-3, 4))              # -3, -2, -1, 0, 1, 2, 3

# typical use: time axis for plotting
HZ = 5
t = np.arange(10) / HZ              # 0.0, 0.2, 0.4, ..., 1.8
print("time axis:", t)

## `np.argmax` — *which index* has the largest value?

Different from `max` (which returns the largest *value*). Used in the synchrony notebook to find which lag had the strongest cross-correlation.

In [ ]:
scores = np.array([0.2, 0.6, 0.9, 0.4, 0.1])
print("largest score:    ", scores.max())
print("position of largest:", np.argmax(scores))   # 2

# real example: find which lag gave the best correlation
lags = np.arange(-2, 3)             # -2, -1, 0, 1, 2
corrs = np.array([0.1, 0.5, 0.3, 0.9, 0.4])
best  = lags[np.argmax(corrs)]      # = 1 (third-from-last is 1)
print("best lag:", best)

## `np.polyfit` — fit a straight line to data

Used in the **injury-prevention** notebook to detect fatigue: fit a line through landing peaks vs. time, positive slope = impacts growing across the session.

In [ ]:
x = np.array([1, 2, 3, 4, 5])
y = np.array([2.1, 3.9, 6.2, 8.1, 9.8])

slope, intercept = np.polyfit(x, y, 1)   # 1 = degree (straight line)
print(f"line: y = {slope:.2f} x + {intercept:.2f}")
# slope ~ 2: y grows by 2 per unit x

---
# Section 3 — scipy.signal: finding patterns in time series

This is the toolkit for `find_peaks` (project 3, project 4) and `correlate` (project 4, project 5).

## `find_peaks` — locate the local maxima

Returns the indices of every value that is *higher than its neighbors*. The `height=` and `distance=` arguments filter out tiny or too-close peaks.

In [ ]:
from scipy.signal import find_peaks

data = [1, 3, 1, 5, 2, 4, 0, 7, 1]

# all peaks above height 2
idx, _ = find_peaks(data, height=2)
print("peak indices:", idx)
print("peak values: ", [data[i] for i in idx])

# require peaks to be at least 3 positions apart
idx, _ = find_peaks(data, height=2, distance=3)
print("with distance=3:", idx)

## `correlate` — how alike are two signals?

Imagine two recordings of the same dance, but one started a fraction of a second later. They have the *same shape*, just shifted in time. How do you find the shift?

**Start with the dot product.** Multiply two arrays element-wise, then add up the results. It's the simplest "how alike?" measure — big when peaks line up, small when they don't:

```
dot([1, 2, 3], [1, 2, 3]) = 1*1 + 2*2 + 3*3 = 14   # identical
dot([1, 2, 3], [3, 2, 1]) = 1*3 + 2*2 + 3*1 = 10   # related but reversed
dot([1, 2, 3], [0, 0, 0]) = 0                       # nothing in common
```

To find the *time shift* between two signals, compute the dot product at **every possible alignment** and pick the largest. That's cross-correlation. Let's build the intuition with a tiny example.

In [ ]:
# b is just a, shifted right by 2 positions: b "lags" a by 2 samples in time.
a = np.array([0, 1, 2, 1, 0, 0, 0])
b = np.array([0, 0, 0, 1, 2, 1, 0])

# At "no shift" -- compare a[i] with b[i] for every i, sum products
print("a:        ", a)
print("b:        ", b)
print("dot(a, b)  =", int(np.sum(a * b)))     # 0+0+0+1+0+0+0 = 1, low

# Try shifting b LEFT by 2 first, then dot product:
b_shifted = np.concatenate([b[2:], [0, 0]])   # [0,1,2,1,0,0,0]  -- now matches a
print("\nb shifted left by 2:", b_shifted)
print("dot(a, b_shifted) =", int(np.sum(a * b_shifted)))   # 0+1+4+1+0+0+0 = 6, high

In [ ]:
from scipy.signal import correlate

# correlate(a, b, "full") computes the dot product at every possible shift
# automatically. Output length = 2*N - 1 for N-length inputs.
xc = correlate(a, b, mode="full")
lags = np.arange(-len(a) + 1, len(a))     # -6, -5, ..., 0, ..., +6

print("lag : dot product")
for lag, val in zip(lags, xc):
    star = "  <-- best match" if val == xc.max() else ""
    print(f"  {lag:+d} : {val}{star}")

best_lag = lags[np.argmax(xc)]
print(f"\nbest lag: {best_lag:+d}    (matches the manual shift above)")

The peak is at **lag = −2** with dot product 6 — exactly what the manual shift gave.

### Reading the sign of the lag

Using the convention `lags = np.arange(-len(a)+1, len(a))`:

| best lag    | meaning                                                  |
|-------------|----------------------------------------------------------|
| **negative**| `b` is **delayed** relative to `a` (b's events come later) |
| **zero**    | `a` and `b` are perfectly aligned                          |
| **positive**| `b` is **earlier** than `a` (b's events come sooner)       |

Our setup: `b = a` shifted right by 2 → `b`'s events occur 2 positions *after* `a`'s. So `b` lags `a`, and the peak is at **lag = −2**. ✓

This is exactly the question the synchrony notebook asks about two dancers: at the best lag, was dancer B delayed (negative) or leading (positive) compared to dancer A?

### Why subtract the mean first?

Real recordings have a constant offset (e.g., gravity making `z` always around −1000 mg). That offset can dominate the dot product even when the shapes don't actually match. The synchrony notebook always does:

```python
xc = correlate(a - a.mean(), b - b.mean(), mode="full")
```

For the toy example above the values are small enough that mean subtraction doesn't change which lag wins — but for real signals, do it.

In [ ]:
import matplotlib.pyplot as plt   # used heavily in section 4; pre-importing here for the plot below

# Visualize: dot product at every shift; the peak is the best alignment.
fig, ax = plt.subplots(figsize=(8, 3))
ax.stem(lags, xc, basefmt=" ")
ax.axvline(best_lag, color="crimson", linestyle="--",
           label=f"best lag = {best_lag:+d}")
ax.set_xlabel("lag")
ax.set_ylabel("dot product")
ax.set_title("correlate(a, b) at every shift")
ax.legend()
plt.show()

---
# Section 4 — matplotlib: plotting basics

Every starter notebook makes plots. The pattern is always: `fig, ax = plt.subplots(...)`, then call methods on `ax`.

In [ ]:
import matplotlib.pyplot as plt

t = np.arange(0, 2*np.pi, 0.1)
y = np.sin(t)

fig, ax = plt.subplots(figsize=(8, 3))   # one plot, 8 inches wide
ax.plot(t, y, color="steelblue", label="sin(t)")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("t")
ax.set_ylabel("sin(t)")
ax.legend()
plt.show()

## Three plot types you'll see all over

- **`ax.plot(x, y)`** — line plot (used for time-series).
- **`ax.scatter(x, y)`** — dots (used when you want to color-code points by category).
- **`ax.hist(values, bins=...)`** — histogram (used in the injury-prevention starter).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# left: line
axes[0].plot(t, np.sin(t), color="steelblue")
axes[0].set_title("plot")

# middle: scatter colored by group
groups = ["A"]*30 + ["B"]*30
x_all = np.concatenate([np.random.normal(0, 1, 30), np.random.normal(3, 1, 30)])
y_all = np.random.normal(0, 1, 60)
for g, color in [("A", "crimson"), ("B", "steelblue")]:
    mask = [grp == g for grp in groups]
    axes[1].scatter(x_all[mask], y_all[mask], color=color, label=g)
axes[1].legend(); axes[1].set_title("scatter")

# right: histogram
axes[2].hist(np.random.normal(100, 15, 500), bins=20, color="crimson", edgecolor="white")
axes[2].set_title("hist")

plt.tight_layout(); plt.show()

## Useful annotations

- **`ax.axhline(value)`** / **`ax.axvline(value)`** — draw a horizontal/vertical reference line. The pose notebook uses these to draw classification thresholds; the injury notebook draws the median-impact line.
- **`ax.legend(loc="upper right", fontsize=8)`** — turn on the legend. Each `label=...` argument from above shows up in it.
- **`ax.set_xlim(...)`**, **`ax.set_ylim(...)`** — clip what's shown. Used in cross-correlation plots so the lag-0 spike doesn't dominate.

---
# Section 5 — sklearn refresher (the L11 recipe)

L11 covered this, but here's the four-line summary in case you skipped it. Used by Project 1 and Project 6.

1. **Split** your features and labels into a *train* set and a *test* set.
2. **Fit** a model on the train set.
3. **Score** it on the test set (the number to trust).
4. *(Optional)* Look at the **confusion matrix** to see *which classes* the model confuses.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

iris = load_iris()
X, y = iris.data, iris.target            # 150 flowers, 4 features each, 3 species

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

# 2. Fit
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

# 3. Score
print("train accuracy:", clf.score(X_train, y_train))
print("test  accuracy:", clf.score(X_test,  y_test))

## `ConfusionMatrixDisplay` — *which* classes does the model confuse?

Rows = true label. Columns = predicted label. The diagonal is correct; anything off-diagonal is a mistake.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap="Blues")
plt.show()

---
# Section 6 — Working in Colab: save your code, save your data

Colab is convenient, but it has two persistence quirks that bite if you don't know about them.

## Quirk 1: opening from GitHub gives you a *read-only* copy

When you click the "Open in Colab" badge on a starter, you get a view of the notebook tied to GitHub. **Editing then closing the tab loses your work.** Solve it once per starter, then forget about it:

> **`File → Save a copy in Drive`**

The copy lands in your Google Drive (`Colab Notebooks/` folder by default). Drive auto-saves on every change. **Bookmark the Drive copy's URL** — that's your working notebook from now on. The GitHub badge always opens the original; if you click it again you're back at the read-only view.

If you want real version control instead of Drive, **`File → Save a copy in GitHub`** does the same thing into your own GitHub repo (you'll need to authorize Colab once). That's an upgrade — Drive is fine for L12.

## Quirk 2: uploaded files vanish when the runtime times out

Files dragged into Colab's *Files* panel — or uploaded via `files.upload()` — live in the **runtime sandbox**. After ~90 minutes idle (or 12 hours total), the runtime resets and the file is gone. **Don't** rely on "I'll re-upload my CSV at the start of each session" — eventually you'll lose data.

The fix: put your CSV in **Google Drive** and mount Drive into Colab.

In [ ]:
# Mount Google Drive into Colab. Run once per session; the first run pops up
# a permission dialog. Skip this cell if you're not in Colab.
#
# After mounting, your Drive shows up under /content/drive/MyDrive/...
# Drop your captured CSVs into a Drive folder (e.g., MyDrive/STEM-on-Stage/)
# and load them like any other path:
#
#     df = pd.read_csv('/content/drive/MyDrive/STEM-on-Stage/microbit-log-jane.csv')
#
# The cell below is commented out so it doesn't run for everyone -- uncomment
# the two lines when you actually want to mount.

# from google.colab import drive
# drive.mount('/content/drive')

## Tracking changes

Three options, in increasing order of fidelity:

- **Drive auto-history** — `File → Revision history` shows ~30 days of saved versions. No diffs, no branches, but trivially easy. Good enough for L12.
- **Periodic `File → Download .ipynb`** — manual local backup. Pair with Drive for end-of-session snapshots.
- **Personal GitHub** — *real* version control with diffs, branches, and a public portfolio. Setup overhead, but the L13–L15 final showcase is a nice first portfolio piece if you want one. Use `File → Save a copy in GitHub` from Colab.

A personal GitHub repo is **not required** for L12 or for the open-topic project — Drive is fine. Treat it as an optional upgrade.

---
# Wrap-up

Every function in the L12 starter notebooks has now appeared above with a tiny example. When you open Project 2/3/4/5/6 and see something unfamiliar, search for it in this notebook (`Ctrl+F` / `Cmd+F`) — and change a value to see how it reacts before going back to the project.

**Reference card.**

| You see…                            | It's from…    | What it does                                          |
|-------------------------------------|---------------|-------------------------------------------------------|
| `pd.read_csv(path)`                 | pandas        | load a CSV into a DataFrame                           |
| `df["col"]`, `df.iloc[a:b]`         | pandas        | pick a column / slice rows                            |
| `df["col"].mean()`, `.std()`, …     | pandas        | per-column statistics                                 |
| `df["col"].apply(fn)`               | pandas        | run a function over every value in a column          |
| `df["col"].value_counts()`          | pandas        | counts per category                                   |
| `np.sqrt(x)`, `x**2`, `x + y`       | numpy         | element-wise math                                     |
| `np.diff(arr)`                      | numpy         | gaps between consecutive values                       |
| `np.arange(start, stop, step)`      | numpy         | a regular sequence as an array                        |
| `np.argmax(arr)`                    | numpy         | *index* of the largest value                          |
| `np.polyfit(x, y, 1)`               | numpy         | fit a straight line, returns (slope, intercept)       |
| `find_peaks(arr, height=, distance=)`| scipy.signal | locate local maxima                                   |
| `correlate(a, b, mode="full")`      | scipy.signal  | cross-correlation between two signals                 |
| `plt.subplots`, `ax.plot/scatter/hist`| matplotlib  | plotting                                              |
| `train_test_split`, `clf.fit`, `clf.score`| sklearn   | the L11 ML recipe                                     |